## Setup

In [1]:
# Standard library imports
import sys
from pathlib import Path
import logging
import json
from datetime import datetime
from decimal import Decimal
from typing import List, Dict, Any, Tuple

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Project imports
from src.agents.decision_agent import (
    DecisionRules,
    StateAggregator,
    DecisionAnalyzer,
    RateCalculator,
    ExplanationGenerator
)
from src.models import (
    ExtractedDocument,
    CreditReport,
    RiskAssessment,
    ComplianceReport,
    LendingDecision,
    PolicyViolation
)
from src.utils.config import Config

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Imports successful")
print(f"📁 Project root: {project_root}")

✅ Imports successful
📁 Project root: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan


---

## Part 1: Initialize Decision Components

### Step 1.1: Create Decision Rules Engine

In [2]:
print("🔧 Initializing DecisionRules...\n")

rules = DecisionRules()

# Display thresholds
thresholds = rules.get_thresholds()
print("📊 Decision Thresholds:")
print(f"  DTI: {thresholds['dti_excellent']:.0f}% (excellent) | {thresholds['dti_acceptable']:.0f}% (acceptable) | {thresholds['dti_max']:.0f}% (max)")
print(f"  LTV: {thresholds['ltv_excellent']:.0f}% (excellent) | {thresholds['ltv_acceptable']:.0f}% (acceptable) | {thresholds['ltv_max']:.0f}% (max)")
print(f"  Credit: {thresholds['credit_excellent']} (excellent) | {thresholds['credit_good']} (good) | {thresholds['credit_near_prime']} (near-prime) | {thresholds['credit_subprime']} (subprime)")
print("\n✅ DecisionRules initialized")

INFO:src.agents.decision_agent:DecisionRules initialized with standard thresholds


🔧 Initializing DecisionRules...

📊 Decision Thresholds:
  DTI: 36% (excellent) | 43% (acceptable) | 50% (max)
  LTV: 80% (excellent) | 95% (acceptable) | 97% (max)
  Credit: 760 (excellent) | 720 (good) | 640 (near-prime) | 580 (subprime)

✅ DecisionRules initialized


### Step 1.2: Create State Aggregator

In [3]:
print("🔧 Initializing StateAggregator...\n")

aggregator = StateAggregator()

print("✅ StateAggregator initialized")
print("📋 Aggregator combines: ExtractedDocument + CreditReport + RiskAssessment + ComplianceReport")

INFO:src.agents.decision_agent:StateAggregator initialized


🔧 Initializing StateAggregator...

✅ StateAggregator initialized
📋 Aggregator combines: ExtractedDocument + CreditReport + RiskAssessment + ComplianceReport


### Step 1.3: Create Decision Analyzer (GPT-4o)

In [4]:
print("🔧 Initializing DecisionAnalyzer...\n")

analyzer = DecisionAnalyzer(
    model=Config.AZURE_OPENAI_DEPLOYMENT_GPT4,
    temperature=0.2  # Consistent decisions
)

print("✅ DecisionAnalyzer initialized")
print(f"🤖 Model: {Config.AZURE_OPENAI_DEPLOYMENT_GPT4}")
print("📊 Temperature: 0.2 (balanced consistency)")

🔧 Initializing DecisionAnalyzer...



INFO:src.agents.decision_agent:DecisionAnalyzer initialized: model=gpt-4o-mini, temperature=0.2


✅ DecisionAnalyzer initialized
🤖 Model: gpt-4o-mini
📊 Temperature: 0.2 (balanced consistency)


### Step 1.4: Create Rate Calculator

In [5]:
print("🔧 Initializing RateCalculator...\n")

rate_calculator = RateCalculator()

print("✅ RateCalculator initialized")
print(f"💰 Base Rate: {rate_calculator.BASE_RATE}%")
print("📊 Pricing Model: Base + Credit Adjustment + Risk Premium + LTV Adjustment")

INFO:src.agents.decision_agent:RateCalculator initialized with base rate: 6.50%


🔧 Initializing RateCalculator...

✅ RateCalculator initialized
💰 Base Rate: 6.50%
📊 Pricing Model: Base + Credit Adjustment + Risk Premium + LTV Adjustment


### Step 1.5: Create Explanation Generator

In [6]:
print("🔧 Initializing ExplanationGenerator...\n")

explainer = ExplanationGenerator(
    model=Config.AZURE_OPENAI_DEPLOYMENT_GPT4,
    temperature=0.3  # Slightly more varied language
)

print("✅ ExplanationGenerator initialized")
print(f"🤖 Model: {Config.AZURE_OPENAI_DEPLOYMENT_GPT4}")
print("📝 Output: Plain-language customer letters (8th grade reading level)")

INFO:src.agents.decision_agent:ExplanationGenerator initialized: model=gpt-4o-mini, temperature=0.3


🔧 Initializing ExplanationGenerator...

✅ ExplanationGenerator initialized
🤖 Model: gpt-4o-mini
📝 Output: Plain-language customer letters (8th grade reading level)


---

## Part 2: Test Data - Create Sample Applications

### Step 2.1: Excellent Application (Auto-Approve)

In [7]:
print("📝 Creating EXCELLENT application (should auto-approve)...\n")

# Extracted document
doc_excellent = ExtractedDocument(
    document_id="DOC-EXCELLENT-001-1",
    application_id="APP-EXCELLENT-001",
    document_type="pay_stub",
    file_path="data/applications/app-excellent-001/paystub.pdf",
    extraction_method="document_intelligence",
    confidence_score=0.95,
    structured_data={
        "applicant_name": "Alice Johnson",
        "ssn": "123-45-6789",
        "employer_name": "Tech Corp Inc",
        "employment_years": 8,
        "annual_income": "125000.00",
        "monthly_income": "10416.00",
        "loan_amount": "300000.00",
        "property_value": "375000.00",
        "loan_purpose": "purchase"
    }
)

# Credit report
credit_excellent = CreditReport(
    ssn="123-45-6789",
    credit_score=780,
    payment_history="excellent",
    credit_utilization=15.0,
    accounts_open=8,
    derogatory_marks=0,
    hard_inquiries_12mo=1,
    credit_age_months=96,
    late_payments_12mo=0
)

# Risk assessment
risk_excellent = RiskAssessment(
    application_id="APP-EXCELLENT-001",
    debt_to_income_ratio=Decimal("32.0"),
    loan_to_value_ratio=Decimal("80.0"),
    monthly_debt_payments=Decimal("3333"),
    monthly_gross_income=Decimal("10416"),
    risk_level="low",
    risk_score=Decimal("18.5"),
    risk_factors=[],
    mitigating_factors=["Excellent credit score", "Low DTI", "Strong income", "Low LTV"],
    recommendation="approve",
    reasoning="Strong financial profile with excellent credit and manageable DTI."
)

# Compliance report
compliance_excellent = ComplianceReport(
    application_id="APP-EXCELLENT-001",
    is_compliant=True,
    compliance_score=Decimal("95.0"),
    violations=[],
    compliance_summary="Application is fully compliant with all underwriting policies and regulatory requirements. No violations detected.",
    policies_evaluated=["credit_requirements.pdf", "underwriting_standards.pdf"],
    rag_chunks_used=5
)

print("✅ Excellent application created:")
print(f"  Credit Score: {credit_excellent.credit_score}")
print(f"  DTI: {risk_excellent.debt_to_income_ratio}%")
print(f"  LTV: {risk_excellent.loan_to_value_ratio}%")
print(f"  Derogatory Marks: {credit_excellent.derogatory_marks}")
print(f"  Compliance Violations: {len(compliance_excellent.violations)}")

📝 Creating EXCELLENT application (should auto-approve)...

✅ Excellent application created:
  Credit Score: 780
  DTI: 32.0%
  LTV: 80.0%
  Derogatory Marks: 0
  Compliance Violations: 0


### Step 2.2: Borderline Application (Conditional Approval)

In [8]:
print("📝 Creating BORDERLINE application (should require GPT-4o analysis)...\n")

# Extracted document
doc_borderline = ExtractedDocument(
    document_id="DOC-BORDERLINE-001-1",
    application_id="APP-BORDERLINE-001",
    document_type="pay_stub",
    file_path="data/applications/app-borderline-001/paystub.pdf",
    extraction_method="document_intelligence",
    confidence_score=0.88,
    structured_data={
        "applicant_name": "Bob Smith",
        "ssn": "234-56-7890",
        "employer_name": "Small Business Inc",
        "employment_years": 4,
        "annual_income": "95000.00",
        "monthly_income": "7916.00",
        "loan_amount": "350000.00",
        "property_value": "400000.00",
        "loan_purpose": "purchase"
    }
)

# Credit report
credit_borderline = CreditReport(
    ssn="234-56-7890",
    credit_score=680,
    payment_history="good",
    credit_utilization=35.0,
    accounts_open=6,
    derogatory_marks=1,
    hard_inquiries_12mo=3,
    credit_age_months=72,
    late_payments_12mo=1
)

# Risk assessment
risk_borderline = RiskAssessment(
    application_id="APP-BORDERLINE-001",
    debt_to_income_ratio=Decimal("39.5"),
    loan_to_value_ratio=Decimal("87.5"),
    monthly_debt_payments=Decimal("3127"),
    monthly_gross_income=Decimal("7916"),
    risk_level="medium",
    risk_score=Decimal("42.3"),
    risk_factors=["Elevated DTI", "High LTV requires PMI", "Recent late payment"],
    mitigating_factors=["Stable employment", "Fair credit score", "Reasonable income"],
    recommendation="review",
    reasoning="Mixed profile with elevated DTI and LTV but acceptable credit score. Requires manual underwriter review."
)

# Compliance report
compliance_borderline = ComplianceReport(
    application_id="APP-BORDERLINE-001",
    is_compliant=True,
    compliance_score=Decimal("78.0"),
    violations=[
        PolicyViolation(
            policy_name="underwriting_standards.pdf",
            policy_section="DTI Guidelines",
            severity="warning",
            description="DTI 39.5% approaches 43% threshold",
            recommendation="Consider compensating factors"
        )
    ],
    compliance_summary="Application is generally compliant but has elevated DTI ratio approaching threshold limits. Manual review recommended.",
    policies_evaluated=["credit_requirements.pdf", "underwriting_standards.pdf"],
    rag_chunks_used=6
)

print("✅ Borderline application created:")
print(f"  Credit Score: {credit_borderline.credit_score}")
print(f"  DTI: {risk_borderline.debt_to_income_ratio}%")
print(f"  LTV: {risk_borderline.loan_to_value_ratio}%")
print(f"  Derogatory Marks: {credit_borderline.derogatory_marks}")
print(f"  Compliance Warnings: {len(compliance_borderline.violations)}")

📝 Creating BORDERLINE application (should require GPT-4o analysis)...

✅ Borderline application created:
  Credit Score: 680
  DTI: 39.5%
  LTV: 87.5%
  Derogatory Marks: 1
  Compliance Warnings: 1


### Step 2.3: Poor Application (Auto-Reject)

In [9]:
print("📝 Creating POOR application (should be auto-rejected)...\n")

# Extracted document
doc_poor = ExtractedDocument(
    document_id="DOC-POOR-001-1",
    application_id="APP-POOR-001",
    document_type="pay_stub",
    file_path="data/applications/app-poor-001/paystub.pdf",
    extraction_method="document_intelligence",
    confidence_score=0.82,
    structured_data={
        "applicant_name": "Charlie Brown",
        "ssn": "345-67-8901",
        "employer_name": "Freelance Consulting",
        "employment_years": 1,
        "annual_income": "48000.00",
        "monthly_income": "4000.00",
        "loan_amount": "320000.00",
        "property_value": "340000.00",
        "loan_purpose": "purchase"
    }
)

# Credit report
credit_poor = CreditReport(
    ssn="345-67-8901",
    credit_score=565,
    payment_history="poor",
    credit_utilization=68.0,
    accounts_open=4,
    derogatory_marks=3,
    hard_inquiries_12mo=6,
    credit_age_months=42,
    late_payments_12mo=4
)

# Risk assessment
risk_poor = RiskAssessment(
    application_id="APP-POOR-001",
    debt_to_income_ratio=Decimal("54.2"),
    loan_to_value_ratio=Decimal("94.1"),
    monthly_debt_payments=Decimal("2168"),
    monthly_gross_income=Decimal("4000"),
    risk_level="high",
    risk_score=Decimal("76.8"),
    risk_factors=[
        "DTI exceeds 43% threshold",
        "Credit score below 580",
        "Recent late payments",
        "High LTV",
        "Short employment history"
    ],
    mitigating_factors=[],
    recommendation="deny",
    reasoning="Multiple disqualifying factors including DTI exceeding 43% threshold, credit score below 580 minimum, and insufficient income relative to loan amount."
)

# Compliance report
compliance_poor = ComplianceReport(
    application_id="APP-POOR-001",
    is_compliant=True,
    compliance_score=Decimal("45.0"),
    violations=[
        PolicyViolation(
            policy_name="underwriting_standards.pdf",
            policy_section="Credit Requirements",
            severity="critical",
            description="Credit score 565 below 580 minimum",
            recommendation="Reject application"
        ),
        PolicyViolation(
            policy_name="underwriting_standards.pdf",
            policy_section="DTI Guidelines",
            severity="critical",
            description="DTI 54.2% exceeds 43% maximum",
            recommendation="Reject application"
        )
    ],
    compliance_summary="Application does not meet minimum underwriting standards for credit score and debt-to-income ratio requirements.",
    policies_evaluated=["credit_requirements.pdf", "underwriting_standards.pdf"],
    rag_chunks_used=8
)

print("✅ Poor application created:")
print(f"  Credit Score: {credit_poor.credit_score}")
print(f"  DTI: {risk_poor.debt_to_income_ratio}%")
print(f"  LTV: {risk_poor.loan_to_value_ratio}%")
print(f"  Derogatory Marks: {credit_poor.derogatory_marks}")
print(f"  Compliance Violations: {len(compliance_poor.violations)}")

📝 Creating POOR application (should be auto-rejected)...

✅ Poor application created:
  Credit Score: 565
  DTI: 54.2%
  LTV: 94.1%
  Derogatory Marks: 3
  Compliance Violations: 2


---

## Part 3: Decision Workflow - Excellent Application

### Step 3.1: Apply Decision Rules

In [10]:
print("="*70)
print("EXCELLENT APPLICATION - Decision Workflow")
print("="*70)
print()

print("Step 1: Applying Decision Rules...\n")

decision_excellent, reasons_excellent, confidence_excellent = rules.evaluate(
    doc_excellent, credit_excellent, risk_excellent, compliance_excellent
)

print(f"Decision: {decision_excellent}")
print(f"Confidence: {confidence_excellent:.2f}")
print(f"\nReasons ({len(reasons_excellent)}):")
for i, reason in enumerate(reasons_excellent, 1):
    print(f"  {i}. {reason}")

INFO:src.agents.decision_agent:Evaluating decision rules for APP-EXCELLENT-001
INFO:src.agents.decision_agent:Key metrics: DTI=32.0%, LTV=80.0%, Credit=780, Derog=0, Compliant=True, CompScore=95.0
INFO:src.agents.decision_agent:AUTO-APPROVE: Excellent profile
INFO:src.agents.decision_agent:Key metrics: DTI=32.0%, LTV=80.0%, Credit=780, Derog=0, Compliant=True, CompScore=95.0
INFO:src.agents.decision_agent:AUTO-APPROVE: Excellent profile


EXCELLENT APPLICATION - Decision Workflow

Step 1: Applying Decision Rules...

Decision: approved
Confidence: 1.00

Reasons (5):
  1. Excellent credit score: 780
  2. Conservative DTI: 32.0%
  3. Strong LTV: 80.0%
  4. No compliance violations
  5. Clean credit history


### Step 3.2: Calculate Interest Rate

In [11]:
print("\nStep 2: Calculating Interest Rate...\n")

if decision_excellent in ["approved", "conditional_approval"]:
    rate_excellent = rate_calculator.calculate(
        credit_score=credit_excellent.credit_score,
        risk_level=risk_excellent.risk_level,
        ltv=float(risk_excellent.loan_to_value_ratio),
        decision=decision_excellent
    )
    
    print(f"APR: {rate_excellent['apr']:.3f}%")
    print(f"Breakdown: {rate_excellent['breakdown']}")
    print(f"Credit Tier: {rate_excellent['tier_classifications']['credit_tier']}")
    
    # Calculate monthly payment
    payment_excellent = rate_calculator.calculate_monthly_payment(
        loan_amount=float(doc_excellent.structured_data["loan_amount"]),
        apr=rate_excellent['apr'],
        term_years=30
    )
    
    print(f"\nMonthly Payment (30-year): ${payment_excellent['monthly_payment']:,.2f}")
    print(f"Total Interest: ${payment_excellent['total_interest']:,.0f}")
    
    # Store for explanation
    rate_excellent['monthly_payment'] = payment_excellent['monthly_payment']
else:
    rate_excellent = None
    print("No rate calculated (application denied)")

INFO:src.agents.decision_agent:Calculating rate: credit=780, risk=low, ltv=80.0%, decision=approved
INFO:src.agents.decision_agent:Calculated APR: 6.000% (breakdown: 6.50% (base) -0.50% (credit) = 6.00%)
INFO:src.agents.decision_agent:Calculated APR: 6.000% (breakdown: 6.50% (base) -0.50% (credit) = 6.00%)



Step 2: Calculating Interest Rate...

APR: 6.000%
Breakdown: 6.50% (base) -0.50% (credit) = 6.00%
Credit Tier: excellent

Monthly Payment (30-year): $1,798.65
Total Interest: $347,514


### Step 3.3: Generate Explanation

In [12]:
print("\nStep 3: Generating Customer Explanation...\n")

# Aggregate state for explanation
state_excellent = aggregator.aggregate(
    doc_excellent, credit_excellent, risk_excellent, compliance_excellent
)

# Generate explanation
explanation_excellent = explainer.generate(
    decision=decision_excellent,
    conditions=[],
    reasoning=risk_excellent.reasoning,
    aggregated_state=state_excellent,
    rate_info=rate_excellent
)

print("📧 DECISION LETTER:")
print("="*70)
print(explanation_excellent['letter'])
print("="*70)

INFO:src.agents.decision_agent:Aggregating state for application APP-EXCELLENT-001
INFO:src.agents.decision_agent:State aggregated: 23 metrics, decision_ready=True
INFO:src.agents.decision_agent:Generating explanation for decision: approved, 0 conditions
INFO:src.agents.decision_agent:State aggregated: 23 metrics, decision_ready=True
INFO:src.agents.decision_agent:Generating explanation for decision: approved, 0 conditions



Step 3: Generating Customer Explanation...



INFO:httpx:HTTP Request: POST https://openai-loan-under-writing.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:src.agents.decision_agent:Generated explanation: 1178 chars, 3 factors
INFO:src.agents.decision_agent:Generated explanation: 1178 chars, 3 factors


📧 DECISION LETTER:
Dear Alice,

We are excited to inform you that your loan application for $300,000 has been approved! This is a significant step towards owning your new home, and we are here to support you through the process.

Your strong financial profile played a big role in this decision. You have an excellent credit score of 780, which shows that you manage your finances well. Additionally, your monthly debt payments are manageable, making you a great candidate for this loan. We also appreciate the value of the property you are purchasing, which is $375,000, providing a solid investment.

You have been approved for an interest rate of 6.000%. This means your estimated monthly payment will be about $1,798.65. We will provide you with all the details you need to finalize the loan and get you into your new home.

Next, we will guide you through the closing process. Please review the loan documents we will send you and prepare to sign them. If you have any questions or need assistan

---

## Part 4: Decision Workflow - Borderline Application

### Step 4.1: Apply Decision Rules

In [13]:
print("="*70)
print("BORDERLINE APPLICATION - Decision Workflow")
print("="*70)
print()

print("Step 1: Applying Decision Rules...\n")

decision_borderline, reasons_borderline, confidence_borderline = rules.evaluate(
    doc_borderline, credit_borderline, risk_borderline, compliance_borderline
)

print(f"Decision: {decision_borderline}")
print(f"Confidence: {confidence_borderline:.2f}")
print(f"\nReasons ({len(reasons_borderline)}):")
for i, reason in enumerate(reasons_borderline, 1):
    print(f"  {i}. {reason}")

INFO:src.agents.decision_agent:Evaluating decision rules for APP-BORDERLINE-001
INFO:src.agents.decision_agent:Key metrics: DTI=39.5%, LTV=87.5%, Credit=680, Derog=1, Compliant=True, CompScore=78.0
INFO:src.agents.decision_agent:BORDERLINE: Requires GPT-4o analysis. Initial direction: lean_conditional
INFO:src.agents.decision_agent:Key metrics: DTI=39.5%, LTV=87.5%, Credit=680, Derog=1, Compliant=True, CompScore=78.0
INFO:src.agents.decision_agent:BORDERLINE: Requires GPT-4o analysis. Initial direction: lean_conditional


BORDERLINE APPLICATION - Decision Workflow

Step 1: Applying Decision Rules...

Decision: refer_to_manual
Confidence: 0.70

Reasons (8):
  1. ⚠ Elevated DTI: 39.5% (above 36.0% but below 43.0%)
  2. ⚠ Elevated LTV: 87.5% (requires PMI)
  3. ⚠ Fair credit: 680
  4. ⚠ 1 derogatory mark(s)
  5. ⚠ Compliant with minor concerns (score: 78)
  6. Risk factors: Elevated DTI, High LTV requires PMI, Recent late payment
  7. Mitigating factors: Stable employment, Fair credit score, Reasonable income
  8. 1 policy warning(s): DTI Guidelines


### Step 4.2: GPT-4o Analysis (Borderline Cases)

In [14]:
print("\nStep 2: GPT-4o Borderline Analysis...\n")

if decision_borderline == "refer_to_manual":
    # Aggregate state
    state_borderline = aggregator.aggregate(
        doc_borderline, credit_borderline, risk_borderline, compliance_borderline
    )
    
    # Analyze with GPT-4o
    print("🤖 Calling GPT-4o for comprehensive analysis...\n")
    
    decision_borderline, conditions_borderline, reasoning_borderline, confidence_borderline = analyzer.analyze(
        aggregated_state=state_borderline,
        initial_direction="lean_conditional",
        rules_reasons=reasons_borderline
    )
    
    print(f"GPT-4o Decision: {decision_borderline}")
    print(f"Confidence: {confidence_borderline:.2f}")
    print(f"\nConditions ({len(conditions_borderline)}):")
    for i, condition in enumerate(conditions_borderline, 1):
        print(f"  {i}. {condition}")
    
    print(f"\nReasoning:\n{reasoning_borderline}")
else:
    conditions_borderline = []
    reasoning_borderline = risk_borderline.reasoning
    state_borderline = aggregator.aggregate(
        doc_borderline, credit_borderline, risk_borderline, compliance_borderline
    )

INFO:src.agents.decision_agent:Aggregating state for application APP-BORDERLINE-001
INFO:src.agents.decision_agent:State aggregated: 23 metrics, decision_ready=True
INFO:src.agents.decision_agent:Analyzing borderline case APP-BORDERLINE-001 with direction: lean_conditional
INFO:src.agents.decision_agent:StateAggregator initialized
INFO:src.agents.decision_agent:State aggregated: 23 metrics, decision_ready=True
INFO:src.agents.decision_agent:Analyzing borderline case APP-BORDERLINE-001 with direction: lean_conditional
INFO:src.agents.decision_agent:StateAggregator initialized



Step 2: GPT-4o Borderline Analysis...

🤖 Calling GPT-4o for comprehensive analysis...



INFO:httpx:HTTP Request: POST https://openai-loan-under-writing.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:src.agents.decision_agent:GPT-4o decision: conditional_approval (confidence: 0.75), 3 conditions
INFO:src.agents.decision_agent:GPT-4o decision: conditional_approval (confidence: 0.75), 3 conditions


GPT-4o Decision: conditional_approval
Confidence: 0.75

Conditions (3):
  1. Obtain PMI due to high LTV of 87.5%
  2. Verify employment stability with Small Business Inc
  3. Maintain reserves equal to at least 2 months of mortgage payments

Reasoning:
The application presents a mixed profile with both strengths and concerns. The applicant has a stable employment history and a reasonable income, which supports their ability to repay the loan. However, the elevated DTI of 39.5% is a concern as it approaches the 43% threshold, and the high LTV of 87.5% necessitates PMI to mitigate risk. The applicant's fair credit score of 680 and one derogatory mark indicate some risk, but the overall payment history is good. Given these factors, conditional approval is warranted with specific conditions to address the risks identified.


### Step 4.3: Calculate Interest Rate

In [15]:
print("\nStep 3: Calculating Interest Rate...\n")

if decision_borderline in ["approved", "conditional_approval"]:
    rate_borderline = rate_calculator.calculate(
        credit_score=credit_borderline.credit_score,
        risk_level=risk_borderline.risk_level,
        ltv=float(risk_borderline.loan_to_value_ratio),
        decision=decision_borderline
    )
    
    print(f"APR: {rate_borderline['apr']:.3f}%")
    print(f"Breakdown: {rate_borderline['breakdown']}")
    print(f"Credit Tier: {rate_borderline['tier_classifications']['credit_tier']}")
    
    # Calculate monthly payment
    payment_borderline = rate_calculator.calculate_monthly_payment(
        loan_amount=float(doc_borderline.structured_data["loan_amount"]),
        apr=rate_borderline['apr'],
        term_years=30
    )
    
    print(f"\nMonthly Payment (30-year): ${payment_borderline['monthly_payment']:,.2f}")
    print(f"Total Interest: ${payment_borderline['total_interest']:,.0f}")
    
    rate_borderline['monthly_payment'] = payment_borderline['monthly_payment']
else:
    rate_borderline = None
    print("No rate calculated (application denied)")

INFO:src.agents.decision_agent:Calculating rate: credit=680, risk=medium, ltv=87.5%, decision=conditional_approval
INFO:src.agents.decision_agent:Calculated APR: 6.900% (breakdown: 6.50% (base) +0.25% (risk) +0.15% (LTV) = 6.90%)
INFO:src.agents.decision_agent:Calculated APR: 6.900% (breakdown: 6.50% (base) +0.25% (risk) +0.15% (LTV) = 6.90%)



Step 3: Calculating Interest Rate...

APR: 6.900%
Breakdown: 6.50% (base) +0.25% (risk) +0.15% (LTV) = 6.90%
Credit Tier: fair

Monthly Payment (30-year): $2,305.10
Total Interest: $479,836


### Step 4.4: Generate Explanation

In [16]:
print("\nStep 4: Generating Customer Explanation...\n")

explanation_borderline = explainer.generate(
    decision=decision_borderline,
    conditions=conditions_borderline,
    reasoning=reasoning_borderline,
    aggregated_state=state_borderline,
    rate_info=rate_borderline
)

print("📧 DECISION LETTER:")
print("="*70)
print(explanation_borderline['letter'])
print("="*70)

INFO:src.agents.decision_agent:Generating explanation for decision: conditional_approval, 3 conditions



Step 4: Generating Customer Explanation...



INFO:httpx:HTTP Request: POST https://openai-loan-under-writing.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:src.agents.decision_agent:Generated explanation: 1587 chars, 5 factors
INFO:src.agents.decision_agent:Generated explanation: 1587 chars, 5 factors


📧 DECISION LETTER:
Dear Bob Smith,

We are excited to inform you that your loan application for $350,000 has been conditionally approved! This means you are on the right path to securing your loan, but we need a few additional items from you to complete the process.

Your application shows some great strengths, like your stable job history and reasonable income, which help show that you can repay the loan. However, we noticed a couple of concerns. Your monthly debt payments are close to the limit we like to see, and the amount you want to borrow is high compared to the value of the property. This means we will need to add Private Mortgage Insurance (PMI) to help protect us. Additionally, your credit score is fair, and we want to ensure everything is in order before we finalize your loan.

To move forward, we need you to complete a few steps. First, please obtain PMI since your loan amount is high compared to the property's value. Second, we need to verify your employment with Small Bus

---

## Part 5: Decision Workflow - Poor Application

### Step 5.1: Apply Decision Rules

In [17]:
print("="*70)
print("POOR APPLICATION - Decision Workflow")
print("="*70)
print()

print("Step 1: Applying Decision Rules...\n")

decision_poor, reasons_poor, confidence_poor = rules.evaluate(
    doc_poor, credit_poor, risk_poor, compliance_poor
)

print(f"Decision: {decision_poor}")
print(f"Confidence: {confidence_poor:.2f}")
print(f"\nReasons ({len(reasons_poor)}):")
for i, reason in enumerate(reasons_poor, 1):
    print(f"  {i}. {reason}")

INFO:src.agents.decision_agent:Evaluating decision rules for APP-POOR-001
INFO:src.agents.decision_agent:Key metrics: DTI=54.2%, LTV=94.1%, Credit=565, Derog=3, Compliant=True, CompScore=45.0
INFO:src.agents.decision_agent:Key metrics: DTI=54.2%, LTV=94.1%, Credit=565, Derog=3, Compliant=True, CompScore=45.0


POOR APPLICATION - Decision Workflow

Step 1: Applying Decision Rules...

Decision: denied
Confidence: 1.00

Reasons (2):
  1. Critical compliance violation: Credit Requirements
  2. Critical compliance violation: DTI Guidelines


### Step 5.2: Generate Explanation (No Rate)

In [18]:
print("\nStep 2: Generating Customer Explanation...\n")

# Aggregate state
state_poor = aggregator.aggregate(
    doc_poor, credit_poor, risk_poor, compliance_poor
)

# Generate explanation
explanation_poor = explainer.generate(
    decision=decision_poor,
    conditions=[],
    reasoning=" | ".join(reasons_poor),
    aggregated_state=state_poor,
    rate_info=None  # No rate for denied applications
)

print("📧 DECISION LETTER:")
print("="*70)
print(explanation_poor['letter'])
print("="*70)

INFO:src.agents.decision_agent:Aggregating state for application APP-POOR-001
INFO:src.agents.decision_agent:State aggregated: 23 metrics, decision_ready=True
INFO:src.agents.decision_agent:Generating explanation for decision: denied, 0 conditions
INFO:src.agents.decision_agent:State aggregated: 23 metrics, decision_ready=True
INFO:src.agents.decision_agent:Generating explanation for decision: denied, 0 conditions



Step 2: Generating Customer Explanation...



INFO:httpx:HTTP Request: POST https://openai-loan-under-writing.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:src.agents.decision_agent:Generated explanation: 1381 chars, 3 factors
INFO:src.agents.decision_agent:Generated explanation: 1381 chars, 3 factors


📧 DECISION LETTER:
Dear Charlie Brown,

Thank you for applying for a loan with us. We appreciate your interest and the time you took to submit your application. After careful review, we regret to inform you that we cannot approve your loan request at this time.

The main reasons for this decision are related to your credit score and your monthly debt payments. Your current credit score is 565, which is below the minimum requirement for this type of loan. Additionally, your monthly debt payments are higher than what we can accept based on our guidelines. These factors are important because they help us ensure that borrowers can comfortably manage their loans.

While we cannot approve your application now, there are steps you can take to improve your situation. We recommend working on increasing your credit score by paying down existing debts and making all your payments on time. This can help you qualify for a loan in the future. You can also review your monthly expenses to see if there

---

## Part 6: Decision Summary Comparison

### Step 6.1: Compare All Three Applications

In [20]:
print("="*70)
print("DECISION SUMMARY - All Applications")
print("="*70)
print()

# Color coding function for decision status
def get_decision_color(decision: str) -> str:
    """Return ANSI color code based on decision status."""
    colors = {
        "approved": "\033[92m",  # Green
        "conditional_approval": "\033[93m",  # Yellow
        "denied": "\033[91m",  # Red
        "refer_to_manual": "\033[93m"  # Yellow
    }
    return colors.get(decision, "\033[0m")  # Default: no color

def format_decision_with_color(decision: str) -> str:
    """Format decision with color coding and emoji indicator."""
    indicators = {
        "approved": "✅",
        "conditional_approval": "⚠️",
        "denied": "❌",
        "refer_to_manual": "🔍"
    }
    color = get_decision_color(decision)
    reset = "\033[0m"
    indicator = indicators.get(decision, "•")
    return f"{color}{indicator} {decision.upper()}{reset}"

def format_confidence_with_color(confidence: float, decision: str) -> str:
    """Format confidence score with color based on level."""
    if confidence >= 0.9:
        color = "\033[92m"  # Green (high confidence)
    elif confidence >= 0.7:
        color = "\033[93m"  # Yellow (medium confidence)
    else:
        color = "\033[91m"  # Red (low confidence)
    reset = "\033[0m"
    return f"{color}{confidence:.2f}{reset}"

summary_data = [
    {
        "Application": "Excellent",
        "Applicant": doc_excellent.structured_data["applicant_name"],
        "Credit": credit_excellent.credit_score,
        "DTI": f"{risk_excellent.debt_to_income_ratio}%",
        "LTV": f"{risk_excellent.loan_to_value_ratio}%",
        "Decision": decision_excellent,
        "Confidence": confidence_excellent,
        "APR": f"{rate_excellent['apr']:.3f}%" if rate_excellent else "N/A"
    },
    {
        "Application": "Borderline",
        "Applicant": doc_borderline.structured_data["applicant_name"],
        "Credit": credit_borderline.credit_score,
        "DTI": f"{risk_borderline.debt_to_income_ratio}%",
        "LTV": f"{risk_borderline.loan_to_value_ratio}%",
        "Decision": decision_borderline,
        "Confidence": confidence_borderline,
        "APR": f"{rate_borderline['apr']:.3f}%" if rate_borderline else "N/A"
    },
    {
        "Application": "Poor",
        "Applicant": doc_poor.structured_data["applicant_name"],
        "Credit": credit_poor.credit_score,
        "DTI": f"{risk_poor.debt_to_income_ratio}%",
        "LTV": f"{risk_poor.loan_to_value_ratio}%",
        "Decision": decision_poor,
        "Confidence": confidence_poor,
        "APR": "N/A"
    }
]

# Print header
print(f"{'Application':<15} {'Applicant':<15} {'Credit':<8} {'DTI':<8} {'LTV':<8} {'Decision':<30} {'Confidence':<12} {'APR':<8}")
print("-" * 115)

# Print rows with color-coded decision and confidence
for row in summary_data:
    decision_display = format_decision_with_color(row['Decision'])
    confidence_display = format_confidence_with_color(row['Confidence'], row['Decision'])
    
    print(f"{row['Application']:<15} {row['Applicant']:<15} {row['Credit']:<8} {row['DTI']:<8} {row['LTV']:<8} {decision_display:<30} {confidence_display:<12} {row['APR']:<8}")

print()
print("\n📊 Decision Legend:")
print("  ✅ APPROVED          - Green (meets all criteria)")
print("  ⚠️  CONDITIONAL      - Yellow (requires additional documentation)")
print("  ❌ DENIED            - Red (does not meet minimum requirements)")
print("  🔍 MANUAL REVIEW     - Yellow (borderline case, needs underwriter)")
print()
print("🎯 Confidence Legend:")
print("  \033[92mHigh (≥0.9)\033[0m - Strong evidence, clear decision")
print("  \033[93mMedium (0.7-0.89)\033[0m - Moderate evidence, some uncertainty")
print("  \033[91mLow (<0.7)\033[0m - Weak evidence, high uncertainty")
print()
print("✅ All decisions completed successfully")

DECISION SUMMARY - All Applications

Application     Applicant       Credit   DTI      LTV      Decision                       Confidence   APR     
-------------------------------------------------------------------------------------------------------------------
Excellent       Alice Johnson   780      32.0%    80.0%    ✅ APPROVED            1.00 6.000%  
Borderline      Bob Smith       680      39.5%    87.5%    ⚠️ CONDITIONAL_APPROVAL 0.75 6.900%  
Poor            Charlie Brown   565      54.2%    94.1%    ❌ DENIED              1.00 N/A     


📊 Decision Legend:
  ✅ APPROVED          - Green (meets all criteria)
  ⚠️  CONDITIONAL      - Yellow (requires additional documentation)
  ❌ DENIED            - Red (does not meet minimum requirements)
  🔍 MANUAL REVIEW     - Yellow (borderline case, needs underwriter)

🎯 Confidence Legend:
  High (≥0.9) - Strong evidence, clear decision
  Medium (0.7-0.89) - Moderate evidence, some uncertainty
  Low (<0.7) - Weak evidence, high uncertainty


### Step 6.2: Decision Confidence Visualization

In [21]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("📊 Creating decision confidence visualization...\n")

# Prepare data for visualization
applications = ["Excellent", "Borderline", "Poor"]
decisions = [decision_excellent, decision_borderline, decision_poor]
confidences = [confidence_excellent, confidence_borderline, confidence_poor]
credits = [credit_excellent.credit_score, credit_borderline.credit_score, credit_poor.credit_score]
dtis = [float(risk_excellent.debt_to_income_ratio), float(risk_borderline.debt_to_income_ratio), float(risk_poor.debt_to_income_ratio)]

# Color mapping for decisions
decision_colors = {
    "approved": "green",
    "conditional_approval": "orange",
    "denied": "red",
    "refer_to_manual": "orange"
}

colors = [decision_colors.get(d, "gray") for d in decisions]

# Create subplots: 1 row, 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Decision Confidence Levels", "Key Metrics Comparison"),
    specs=[[{"type": "bar"}, {"type": "scatter"}]]
)

# Subplot 1: Confidence bar chart
fig.add_trace(
    go.Bar(
        x=applications,
        y=confidences,
        marker=dict(
            color=colors,
            line=dict(color='darkgray', width=2)
        ),
        text=[f"{c:.2%}" for c in confidences],
        textposition='outside',
        name="Confidence",
        hovertemplate="<b>%{x}</b><br>" +
                      "Confidence: %{y:.2%}<br>" +
                      "Decision: %{customdata}<br>" +
                      "<extra></extra>",
        customdata=decisions
    ),
    row=1, col=1
)

# Add confidence threshold lines
fig.add_hline(y=0.9, line_dash="dash", line_color="green", 
              annotation_text="High Confidence (≥0.9)", 
              annotation_position="right",
              row=1, col=1)
fig.add_hline(y=0.7, line_dash="dash", line_color="orange", 
              annotation_text="Medium Confidence (≥0.7)", 
              annotation_position="right",
              row=1, col=1)

# Subplot 2: Credit Score vs DTI scatter
fig.add_trace(
    go.Scatter(
        x=credits,
        y=dtis,
        mode='markers+text',
        marker=dict(
            size=20,
            color=colors,
            line=dict(color='darkgray', width=2),
            symbol=['circle', 'diamond', 'square']
        ),
        text=applications,
        textposition="top center",
        name="Applications",
        hovertemplate="<b>%{text}</b><br>" +
                      "Credit Score: %{x}<br>" +
                      "DTI: %{y:.1f}%<br>" +
                      "<extra></extra>"
    ),
    row=1, col=2
)

# Add threshold lines for credit and DTI
fig.add_vline(x=720, line_dash="dash", line_color="green", 
              annotation_text="Excellent Credit (≥720)", 
              annotation_position="top",
              row=1, col=2)
fig.add_vline(x=640, line_dash="dash", line_color="orange", 
              annotation_text="Near-Prime (≥640)", 
              annotation_position="bottom",
              row=1, col=2)
fig.add_hline(y=36, line_dash="dash", line_color="green", 
              annotation_text="Excellent DTI (≤36%)", 
              annotation_position="left",
              row=1, col=2)
fig.add_hline(y=43, line_dash="dash", line_color="red", 
              annotation_text="Max DTI (43%)", 
              annotation_position="left",
              row=1, col=2)

# Update layout
fig.update_xaxes(title_text="Application", row=1, col=1)
fig.update_yaxes(title_text="Confidence Score", range=[0, 1.1], row=1, col=1)
fig.update_xaxes(title_text="Credit Score", range=[550, 800], row=1, col=2)
fig.update_yaxes(title_text="Debt-to-Income Ratio (%)", range=[25, 60], row=1, col=2)

fig.update_layout(
    title_text="Decision Analysis Dashboard",
    showlegend=False,
    height=500,
    width=1200,
    template="plotly_white"
)

fig.show()

print("✅ Visualization complete")

📊 Creating decision confidence visualization...



✅ Visualization complete


---

## Part 7: Key Insights

### What We Learned

1. **Decision Rules**: Deterministic logic provides consistency
   - Auto-approve: Excellent profiles (DTI≤36%, LTV≤80%, credit≥720)
   - Auto-reject: Poor profiles (DTI>43% + credit<640, credit<580, etc.)
   - Borderline: GPT-4o analysis for nuanced cases

2. **State Aggregation**: Combines all agent outputs
   - 43 key metrics extracted
   - Validation of required fields
   - Human-readable formatting for prompts

3. **GPT-4o Analysis**: Handles complexity
   - 5-step analysis framework
   - Considers compensating factors
   - Identifies specific conditions
   - Temperature 0.2 for consistency

4. **Risk-Adjusted Rates**: Transparent pricing
   - Base rate + adjustments
   - Credit tiers (excellent → subprime)
   - Risk premiums (low/medium/high)
   - LTV adjustments (conventional → high LTV)

5. **Plain-Language Explanations**: Customer-friendly
   - 8th grade reading level
   - ECOA compliance (adverse action notices)
   - Actionable next steps
   - Timeline expectations

### Complete Decision Flow

```
Application Data
  ↓
StateAggregator.aggregate()
  ↓
DecisionRules.evaluate()
  ↓
├─ Auto-Approve → RateCalculator → ExplanationGenerator
├─ Auto-Reject → ExplanationGenerator (no rate)
└─ Borderline → DecisionAnalyzer (GPT-4o)
                  ↓
              RateCalculator → ExplanationGenerator
```

### Next Steps

- **Phase 7**: Multi-agent orchestration with LangGraph
- **Phase 8**: MLflow experiment tracking
- **Phase 9**: Cost optimization analysis

---

**✅ Phase 6 Complete**: Decision Agent fully functional with transparent, compliant lending decisions!